In [1]:
# Mount Google Drive (skip if already mounted)
import os
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted.')
else:
    from google.colab import drive
    drive.mount('/content/drive')

"""
=============================================================================
BreakHis Final Paper Analysis
=============================================================================
Title: Reliability-Aware Multi-Magnification Aggregation for Patient-Level
       Breast Histopathology Classification

Author: Nisreen Albzour | Binghamton University

Purpose:
    Reproduce and verify every result reported in the manuscript from
    already-saved prediction files. Generate all publication-quality
    figures and tables. No models are loaded, trained, or fine-tuned.
    No images are downloaded or processed.

    All fixed-split results are EXPLORATORY. Patient-aware cross-validation
    is a planned confirmatory analysis but is NOT run here.
    No clinical validation is claimed.
=============================================================================
"""

import os
import sys
import json
import time
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    recall_score, confusion_matrix, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')

START_TIME = time.time()

# ============================================================
# Configuration
# ============================================================
RESULTS_DIR = "/content/drive/MyDrive/BreakHis_Project/results"
OUTPUT_DIR  = "/content/drive/MyDrive/BreakHis_Project/final_paper_outputs"

RANDOM_SEED    = 42
THRESHOLD      = 0.5
N_BOOTSTRAP    = 500
MAGNIFICATIONS = ['40X', '100X', '200X', '400X']

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'serif',
})

GENERATED = []
SKIPPED   = []

print("=" * 70)
print("BreakHis Final Paper Analysis")
print("=" * 70)
print(f"  Results dir: {RESULTS_DIR}")
print(f"  Output dir:  {OUTPUT_DIR}")
print(f"  Bootstrap:   {N_BOOTSTRAP} resamples, seed {RANDOM_SEED}")
print(f"  Threshold:   {THRESHOLD}")
print()

# ============================================================
# Discover available files
# ============================================================
print("Available result files:")
if os.path.isdir(RESULTS_DIR):
    for f in sorted(os.listdir(RESULTS_DIR)):
        sz = os.path.getsize(os.path.join(RESULTS_DIR, f))
        print(f"  {f:60s} ({sz:>10,} bytes)")
else:
    raise SystemExit(f"Results directory not found: {RESULTS_DIR}")


Mounted at /content/drive
BreakHis Final Paper Analysis
  Results dir: /content/drive/MyDrive/BreakHis_Project/results
  Output dir:  /content/drive/MyDrive/BreakHis_Project/final_paper_outputs
  Bootstrap:   500 resamples, seed 42
  Threshold:   0.5

Available result files:
  breakhis_audit_summary.txt                                   (     1,905 bytes)
  densenet121_confusion_matrix.csv                             (        69 bytes)
  densenet121_patient_level_metrics.json                       (     1,373 bytes)
  densenet121_patient_level_predictions.csv                    (       968 bytes)
  densenet121_permagnification_metrics.csv                     (     1,284 bytes)
  densenet121_test_metrics.json                                (     1,246 bytes)
  densenet121_training_history.csv                             (     1,952 bytes)
  densenet121_val_test_predictions.csv                         (   554,342 bytes)
  efficientnet_b0_confusion_matrix.csv                         (    

In [2]:
"""
=============================================================================
SECTION 1: Metric Functions and Manuscript Verification Framework
=============================================================================
"""

# ============================================================
# Core metric functions
# ============================================================
def safe_nll(yt, yp):
    eps = 1e-12
    p = np.clip(yp, eps, 1 - eps)
    return float(-np.mean(yt * np.log(p) + (1 - yt) * np.log(1 - p)))


def compute_ece(yt, yp, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (yp >= lo) & (yp < hi) if i < n_bins - 1 else (yp >= lo) & (yp <= hi)
        nb = mask.sum()
        if nb == 0:
            continue
        ece += (nb / len(yt)) * abs(yt[mask].mean() - yp[mask].mean())
    return ece


def full_metrics(yt, yp, threshold=0.5):
    """Compute the full metric suite used in the manuscript."""
    ypr = (yp >= threshold).astype(int)
    both = len(np.unique(yt)) > 1
    return {
        'accuracy':          accuracy_score(yt, ypr),
        'balanced_accuracy': balanced_accuracy_score(yt, ypr),
        'macro_f1':          f1_score(yt, ypr, average='macro', zero_division=0),
        'sensitivity':       recall_score(yt, ypr, pos_label=1, zero_division=0),
        'specificity':       recall_score(yt, ypr, pos_label=0, zero_division=0),
        'roc_auc':           roc_auc_score(yt, yp) if both else None,
        'auprc':             average_precision_score(yt, yp) if both else None,
        'brier':             float(brier_score_loss(yt, yp)),
        'nll':               safe_nll(yt, yp),
        'ece':               compute_ece(yt, yp),
    }


def verify_against_manuscript(name, computed, expected, tol=0.001):
    """Compare computed metrics to manuscript values. Warn on mismatch."""
    mismatches = []
    for key, exp_val in expected.items():
        comp_val = computed.get(key)
        if comp_val is None or exp_val is None:
            continue
        diff = abs(comp_val - exp_val)
        if diff > tol:
            mismatches.append(f"  WARNING: {key}: computed={comp_val:.4f} "
                              f"vs manuscript={exp_val:.4f} (diff={diff:.4f})")
    if mismatches:
        print(f"\n  MISMATCHES for {name}:")
        for m in mismatches:
            print(m)
    else:
        print(f"  {name}: all metrics match manuscript (tol={tol})")
    return len(mismatches) == 0


# ============================================================
# Manuscript expected values (Tables 2 and 4)
# ============================================================
MANUSCRIPT_BASELINES = {
    'resnet50':       {'accuracy': 0.9091, 'balanced_accuracy': 0.8333,
                       'macro_f1': 0.8706, 'roc_auc': 0.9583, 'auprc': 0.9861,
                       'brier': 0.0793, 'nll': 0.2642, 'ece': 0.0878},
    'efficientnet_b0':{'accuracy': 0.9091, 'balanced_accuracy': 0.8333,
                       'macro_f1': 0.8706, 'roc_auc': 0.9583, 'auprc': 0.9861,
                       'brier': 0.0721, 'nll': 0.2312, 'ece': 0.0733},
    'densenet121':    {'accuracy': 0.9091, 'balanced_accuracy': 0.8333,
                       'macro_f1': 0.8706, 'roc_auc': 1.0000, 'auprc': 1.0000,
                       'brier': 0.0550, 'nll': 0.1672, 'ece': 0.1067},
    'swin_small':     {'accuracy': 0.9091, 'balanced_accuracy': 0.8333,
                       'macro_f1': 0.8706, 'roc_auc': 1.0000, 'auprc': 1.0000,
                       'brier': 0.0620, 'nll': 0.2052, 'ece': 0.1384},
}

MANUSCRIPT_LEARNED_VAL = {
    'accuracy': 0.9231, 'balanced_accuracy': 0.9444, 'macro_f1': 0.9150,
    'sensitivity': 0.8889, 'specificity': 1.0000, 'roc_auc': 0.9444,
    'brier': 0.2382, 'nll': 0.6695,
}

MANUSCRIPT_LEARNED_TEST = {
    'accuracy': 1.0000, 'balanced_accuracy': 1.0000, 'macro_f1': 1.0000,
    'sensitivity': 1.0000, 'specificity': 1.0000, 'roc_auc': 1.0000,
    'brier': 0.2387, 'nll': 0.6705,
}

print("Metric functions defined. Manuscript expected values loaded.")


Metric functions defined. Manuscript expected values loaded.


In [3]:
"""
=============================================================================
SECTION 2: Load Baseline Predictions and Verify Manuscript Table 2
=============================================================================
"""

print("=" * 70)
print("SECTION 2: Baseline Patient-Level Verification (Table 2)")
print("=" * 70)

MODELS = {
    'resnet50':        'ResNet50',
    'efficientnet_b0': 'EfficientNet-B0',
    'densenet121':     'DenseNet121',
    'swin_small':      'Swin-Small',
}

# Probability column name in the patient-level prediction CSVs
PROB_COL = 'mean_probability_malignant'

baseline_results = {}

for prefix, label in MODELS.items():
    path = os.path.join(RESULTS_DIR, f"{prefix}_patient_level_predictions.csv")
    if not os.path.exists(path):
        print(f"\n  SKIPPING {label}: file not found at {path}")
        SKIPPED.append(f"{prefix}_patient_level_predictions.csv")
        continue

    df = pd.read_csv(path)
    print(f"\n  {label}: loaded {len(df)} rows")
    print(f"    Columns: {list(df.columns)}")

    # Validate: one true label per patient
    if 'patient_id' in df.columns and 'true_label' in df.columns:
        dup = df.groupby('patient_id')['true_label'].nunique()
        inconsistent = dup[dup > 1]
        if len(inconsistent) > 0:
            print(f"    ERROR: inconsistent labels for {list(inconsistent.index)}")
            continue
        print(f"    Label consistency: OK (all patients have one label)")

    # Test split only
    if 'split' in df.columns:
        test = df[df['split'] == 'test']
    else:
        test = df  # assume all are test if no split column
    print(f"    Test patients: {len(test)}")

    yt = test['true_label'].values
    yp = test[PROB_COL].values
    m = full_metrics(yt, yp)
    baseline_results[prefix] = m

    # Print
    for k, v in m.items():
        print(f"    {k:22s}: {v:.4f}" if v is not None else f"    {k:22s}: n/a")

    # Verify
    verify_against_manuscript(label, m, MANUSCRIPT_BASELINES.get(prefix, {}))


# Save Table 2
if baseline_results:
    rows = []
    for prefix, label in MODELS.items():
        if prefix not in baseline_results:
            continue
        m = baseline_results[prefix]
        rows.append({'Model': label, **m})
    table2 = pd.DataFrame(rows)
    p = os.path.join(OUTPUT_DIR, "table2_baseline_comparison.csv")
    table2.to_csv(p, index=False)
    GENERATED.append("table2_baseline_comparison.csv")
    print(f"\n  Saved: {p}")


SECTION 2: Baseline Patient-Level Verification (Table 2)

  ResNet50: loaded 24 rows
    Columns: ['split', 'patient_id', 'true_label', 'mean_probability_malignant', 'n_images', 'predicted_class']
    Label consistency: OK (all patients have one label)
    Test patients: 11
    accuracy              : 0.9091
    balanced_accuracy     : 0.8333
    macro_f1              : 0.8706
    sensitivity           : 1.0000
    specificity           : 0.6667
    roc_auc               : 0.9583
    auprc                 : 0.9861
    brier                 : 0.0793
    nll                   : 0.2642
    ece                   : 0.0908

  MISMATCHES for ResNet50:

  EfficientNet-B0: loaded 24 rows
    Columns: ['split', 'patient_id', 'true_label', 'mean_probability_malignant', 'n_images', 'predicted_class']
    Label consistency: OK (all patients have one label)
    Test patients: 11
    accuracy              : 0.9091
    balanced_accuracy     : 0.8333
    macro_f1              : 0.8706
    sensitivity  

In [4]:
"""
=============================================================================
SECTION 3: Learned Aggregator Verification (Table 4)
=============================================================================
"""

print("=" * 70)
print("SECTION 3: Learned Aggregator Verification (Table 4)")
print("=" * 70)

s3c_path = os.path.join(RESULTS_DIR, "stage3c_patient_predictions.csv")

if not os.path.exists(s3c_path):
    print(f"  SKIPPING: {s3c_path} not found")
    SKIPPED.append("stage3c_patient_predictions.csv")
else:
    s3c = pd.read_csv(s3c_path)
    print(f"  Loaded: {len(s3c)} rows")
    print(f"  Columns: {list(s3c.columns)}")

    learned_results = {}
    for split in ['val', 'test']:
        sub = s3c[s3c['split'] == split]
        if len(sub) == 0:
            print(f"  {split}: no rows")
            continue
        yt = sub['true_label'].values
        yp = sub['predicted_probability'].values
        m = full_metrics(yt, yp)
        learned_results[split] = m

        print(f"\n  {split.upper()} (n={len(sub)}):")
        for k, v in m.items():
            print(f"    {k:22s}: {v:.4f}" if v is not None else f"    {k:22s}: n/a")

        expected = MANUSCRIPT_LEARNED_VAL if split == 'val' else MANUSCRIPT_LEARNED_TEST
        verify_against_manuscript(f"Learned ({split})", m, expected)

    # Save Table 4
    rows = []
    for split in ['train', 'val', 'test']:
        sub = s3c[s3c['split'] == split]
        if len(sub) == 0:
            continue
        yt = sub['true_label'].values
        yp = sub['predicted_probability'].values
        m = full_metrics(yt, yp)
        tag = " (biased)" if split == 'train' else ""
        rows.append({'Partition': split.title() + tag, 'n': len(sub), **m})

    table4 = pd.DataFrame(rows)
    p = os.path.join(OUTPUT_DIR, "table4_learned_aggregator.csv")
    table4.to_csv(p, index=False)
    GENERATED.append("table4_learned_aggregator.csv")
    print(f"\n  Saved: {p}")


SECTION 3: Learned Aggregator Verification (Table 4)
  Loaded: 82 rows
  Columns: ['split', 'patient_id', 'true_label', 'predicted_probability', 'predicted_class']

  VAL (n=13):
    accuracy              : 0.9231
    balanced_accuracy     : 0.9444
    macro_f1              : 0.9150
    sensitivity           : 0.8889
    specificity           : 1.0000
    roc_auc               : 0.9444
    auprc                 : 0.9798
    brier                 : 0.2382
    nll                   : 0.6695
    ece                   : 0.2590
  Learned (val): all metrics match manuscript (tol=0.001)

  TEST (n=11):
    accuracy              : 1.0000
    balanced_accuracy     : 1.0000
    macro_f1              : 1.0000
    sensitivity           : 1.0000
    specificity           : 1.0000
    roc_auc               : 1.0000
    auprc                 : 1.0000
    brier                 : 0.2387
    nll                   : 0.6705
    ece                   : 0.3978
  Learned (test): all metrics match manuscript 

In [5]:
"""
=============================================================================
SECTION 4: Bootstrap Confidence Intervals (500 resamples)
=============================================================================
"""

print("=" * 70)
print(f"SECTION 4: Bootstrap CIs ({N_BOOTSTRAP} resamples)")
print("=" * 70)

boot_all = []

# Baselines (test split)
for prefix, label in MODELS.items():
    path = os.path.join(RESULTS_DIR, f"{prefix}_patient_level_predictions.csv")
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    test = df[df['split'] == 'test'] if 'split' in df.columns else df
    yt = test['true_label'].values
    yp = test[PROB_COL].values
    point = full_metrics(yt, yp)
    rng = np.random.RandomState(RANDOM_SEED)
    n = len(yt)
    boot_vals = {k: [] for k in point}
    for _ in range(N_BOOTSTRAP):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(yt[idx])) < 2:
            continue
        bm = full_metrics(yt[idx], yp[idx])
        for k in boot_vals:
            if bm[k] is not None:
                boot_vals[k].append(bm[k])
    for k, vals in boot_vals.items():
        if len(vals) == 0:
            continue
        boot_all.append({
            'model': label, 'metric': k,
            'point': point[k],
            'ci_lo': np.percentile(vals, 2.5),
            'ci_hi': np.percentile(vals, 97.5),
            'n_boot': len(vals),
        })

# Learned aggregator (test split)
if os.path.exists(s3c_path):
    sub = s3c[s3c['split'] == 'test']
    yt = sub['true_label'].values
    yp = sub['predicted_probability'].values
    point = full_metrics(yt, yp)
    rng = np.random.RandomState(RANDOM_SEED)
    n = len(yt)
    boot_vals = {k: [] for k in point}
    for _ in range(N_BOOTSTRAP):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(yt[idx])) < 2:
            continue
        bm = full_metrics(yt[idx], yp[idx])
        for k in boot_vals:
            if bm[k] is not None:
                boot_vals[k].append(bm[k])
    for k, vals in boot_vals.items():
        if len(vals) == 0:
            continue
        boot_all.append({
            'model': 'Learned aggregator', 'metric': k,
            'point': point[k],
            'ci_lo': np.percentile(vals, 2.5),
            'ci_hi': np.percentile(vals, 97.5),
            'n_boot': len(vals),
        })

if boot_all:
    boot_df = pd.DataFrame(boot_all)
    p = os.path.join(OUTPUT_DIR, "table_bootstrap_cis.csv")
    boot_df.to_csv(p, index=False)
    GENERATED.append("table_bootstrap_cis.csv")
    print(f"  Saved: {p} ({len(boot_df)} rows)")
    print(f"\n  Key CIs (test, n=11, exploratory):")
    for model in boot_df['model'].unique():
        sub = boot_df[boot_df['model'] == model]
        f1_row = sub[sub['metric'] == 'macro_f1']
        if len(f1_row) > 0:
            r = f1_row.iloc[0]
            print(f"    {model:25s} F1={r['point']:.3f} [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]")
    print("\n  Note: With 11 test patients, CIs are very wide.")


SECTION 4: Bootstrap CIs (500 resamples)
  Saved: /content/drive/MyDrive/BreakHis_Project/final_paper_outputs/table_bootstrap_cis.csv (50 rows)

  Key CIs (test, n=11, exploratory):
    ResNet50                  F1=0.871 [0.450, 1.000]
    EfficientNet-B0           F1=0.871 [0.450, 1.000]
    DenseNet121               F1=0.871 [0.450, 1.000]
    Swin-Small                F1=0.871 [0.450, 1.000]
    Learned aggregator        F1=1.000 [1.000, 1.000]

  Note: With 11 test patients, CIs are very wide.


In [6]:
"""
=============================================================================
SECTION 5: Confusion Matrices (ResNet50, Swin-Small, Learned Aggregator)
=============================================================================
"""

print("=" * 70)
print("SECTION 5: Patient-Level Confusion Matrices")
print("=" * 70)

def plot_confusion_matrix(yt, yp, title, ax):
    ypr = (yp >= THRESHOLD).astype(int)
    cm = confusion_matrix(yt, ypr, labels=[0, 1])
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=max(cm.max(), 1))
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Benign', 'Malignant'])
    ax.set_yticklabels(['Benign', 'Malignant'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontsize=11)
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color=color, fontsize=18, fontweight='bold')


panels = []  # (title, yt, yp)

for prefix, label in [('resnet50', 'ResNet50'), ('swin_small', 'Swin-Small')]:
    path = os.path.join(RESULTS_DIR, f"{prefix}_patient_level_predictions.csv")
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    test = df[df['split'] == 'test'] if 'split' in df.columns else df
    panels.append((label, test['true_label'].values,
                    test[PROB_COL].values))

if os.path.exists(s3c_path):
    sub = s3c[s3c['split'] == 'test']
    panels.append(('Learned Aggregator', sub['true_label'].values,
                    sub['predicted_probability'].values))

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.5))
    if len(panels) == 1:
        axes = [axes]
    for ax, (title, yt, yp) in zip(axes, panels):
        plot_confusion_matrix(yt, yp, title, ax)
    fig.suptitle('Patient-Level Confusion Matrices (Test, n=11)\n'
                 'Exploratory fixed-split evaluation',
                 fontsize=12, y=1.04)
    fig.tight_layout()
    for ext in ['png', 'pdf']:
        p = os.path.join(OUTPUT_DIR, f"fig_confusion_matrices.{ext}")
        fig.savefig(p)
    plt.close(fig)
    GENERATED.append("fig_confusion_matrices.png/pdf")
    print(f"  Saved confusion matrices ({len(panels)} panels)")
else:
    SKIPPED.append("fig_confusion_matrices")
    print("  Skipped: no prediction files found")


SECTION 5: Patient-Level Confusion Matrices
  Saved confusion matrices (3 panels)


In [7]:
"""
=============================================================================
SECTION 6: Reliability Diagrams
=============================================================================
5 bins because n=11 test patients.
"""

print("=" * 70)
print("SECTION 6: Reliability Diagrams")
print("=" * 70)


def plot_reliability(yt, yp, title, ax, n_bins=5, extra_text=""):
    """Reliability diagram with bin counts, 5 bins for small n."""
    edges = np.linspace(0, 1, n_bins + 1)
    centers, accs, counts = [], [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (yp >= lo) & (yp < hi) if i < n_bins - 1 else (yp >= lo) & (yp <= hi)
        nb = mask.sum()
        if nb > 0:
            centers.append(yp[mask].mean())
            accs.append(yt[mask].mean())
            counts.append(nb)

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
    if centers:
        bars = ax.bar(centers, accs, width=1.0 / n_bins, alpha=0.6,
                       edgecolor='black', align='center', color='#2E86AB')
        ax.plot(centers, accs, 'ro-', ms=5, zorder=5)
        for c, a, ct in zip(centers, accs, counts):
            ax.annotate(f'n={ct}', (c, a + 0.05), ha='center', fontsize=8)

    ece = compute_ece(yt, yp, n_bins=n_bins)
    brier = float(brier_score_loss(yt, yp))
    nll = safe_nll(yt, yp)
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Observed Fraction Malignant')
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.15)
    ax.legend(loc='lower right', fontsize=8)
    ax.text(0.02, 1.05, f'ECE={ece:.3f}  Brier={brier:.3f}  NLL={nll:.3f}',
            fontsize=7, transform=ax.transData)
    if extra_text:
        ax.text(0.02, 0.95, extra_text, fontsize=7, transform=ax.transData,
                style='italic', color='gray')


# --- Before / after temperature scaling for ResNet50 and Swin-Small ---
fig_made = False
for prefix, label in [('resnet50', 'ResNet50'), ('swin_small', 'Swin-Small')]:
    rel_path = os.path.join(RESULTS_DIR, f"{prefix}_reliability_predictions.csv")
    if not os.path.exists(rel_path):
        SKIPPED.append(f"fig_reliability_{prefix}")
        continue

    rel_df = pd.read_csv(rel_path)
    test_rel = rel_df[rel_df['split'] == 'test']
    if len(test_rel) == 0:
        continue

    # Aggregate to patient level (before and after calibration)
    uncal_col = 'uncalibrated_probability_malignant'
    cal_col = 'calibrated_probability_malignant'

    if uncal_col not in test_rel.columns or cal_col not in test_rel.columns:
        print(f"  {label}: missing probability columns, skipping")
        continue

    pat_before = test_rel.groupby('patient_id').agg(
        yt=('true_label', 'first'),
        yp=(uncal_col, 'mean'),
    ).reset_index()
    pat_after = test_rel.groupby('patient_id').agg(
        yt=('true_label', 'first'),
        yp=(cal_col, 'mean'),
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    plot_reliability(pat_before['yt'].values, pat_before['yp'].values,
                     f'{label} Before Temperature Scaling', axes[0])
    plot_reliability(pat_after['yt'].values, pat_after['yp'].values,
                     f'{label} After Temperature Scaling', axes[1])
    fig.suptitle(f'{label}: Patient-Level Reliability (Test, n=11)\n'
                 f'Exploratory fixed-split evaluation',
                 fontsize=11, y=1.03)
    fig.tight_layout()
    for ext in ['png', 'pdf']:
        p = os.path.join(OUTPUT_DIR, f"fig_reliability_{prefix}.{ext}")
        fig.savefig(p)
    plt.close(fig)
    GENERATED.append(f"fig_reliability_{prefix}.png/pdf")
    print(f"  Saved: {label} reliability diagrams")
    fig_made = True


# --- Learned aggregator reliability ---
if os.path.exists(s3c_path):
    sub = s3c[s3c['split'] == 'test']
    if len(sub) > 0:
        fig, ax = plt.subplots(figsize=(5, 4.5))
        plot_reliability(sub['true_label'].values,
                         sub['predicted_probability'].values,
                         'Learned Reliability-Aware Aggregator', ax)
        fig.suptitle('Patient-Level Reliability (Test, n=11)\n'
                     'Exploratory fixed-split evaluation',
                     fontsize=11, y=1.03)
        fig.tight_layout()
        for ext in ['png', 'pdf']:
            p = os.path.join(OUTPUT_DIR, f"fig_reliability_learned.{ext}")
            fig.savefig(p)
        plt.close(fig)
        GENERATED.append("fig_reliability_learned.png/pdf")
        print("  Saved: Learned aggregator reliability diagram")


SECTION 6: Reliability Diagrams
  Saved: ResNet50 reliability diagrams
  Saved: Swin-Small reliability diagrams
  Saved: Learned aggregator reliability diagram


In [8]:
"""
=============================================================================
SECTION 7: ROC and Precision-Recall Curves (Four Baselines)
=============================================================================
"""

print("=" * 70)
print("SECTION 7: ROC and Precision-Recall Curves")
print("=" * 70)

colors = {'resnet50': '#2E86AB', 'efficientnet_b0': '#A23B72',
          'densenet121': '#F18F01', 'swin_small': '#C73E1D'}

roc_data = {}
pr_data = {}

for prefix, label in MODELS.items():
    path = os.path.join(RESULTS_DIR, f"{prefix}_patient_level_predictions.csv")
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    test = df[df['split'] == 'test'] if 'split' in df.columns else df
    yt = test['true_label'].values
    yp = test[PROB_COL].values
    if len(np.unique(yt)) < 2:
        continue
    fpr, tpr, _ = roc_curve(yt, yp)
    prec, rec, _ = precision_recall_curve(yt, yp)
    auc_val = roc_auc_score(yt, yp)
    ap_val = average_precision_score(yt, yp)
    roc_data[prefix] = (fpr, tpr, auc_val, label)
    pr_data[prefix] = (rec, prec, ap_val, label)

if roc_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for prefix in MODELS:
        if prefix not in roc_data:
            continue
        fpr, tpr, auc_val, label = roc_data[prefix]
        ax1.plot(fpr, tpr, color=colors[prefix], lw=1.5,
                 label=f'{label} (AUC={auc_val:.3f})')
    ax1.plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('Patient-Level ROC Curves')
    ax1.legend(loc='lower right')

    for prefix in MODELS:
        if prefix not in pr_data:
            continue
        rec, prec, ap_val, label = pr_data[prefix]
        ax2.plot(rec, prec, color=colors[prefix], lw=1.5,
                 label=f'{label} (AP={ap_val:.3f})')
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Patient-Level Precision-Recall Curves')
    ax2.legend(loc='lower left')

    fig.suptitle('Four-Model Comparison (Test, n=11 patients)\n'
                 'Exploratory fixed-split evaluation', fontsize=11, y=1.03)
    fig.tight_layout()
    for ext in ['png', 'pdf']:
        p = os.path.join(OUTPUT_DIR, f"fig_roc_pr_curves.{ext}")
        fig.savefig(p)
    plt.close(fig)
    GENERATED.append("fig_roc_pr_curves.png/pdf")
    print("  Saved: ROC and PR curves")
else:
    SKIPPED.append("fig_roc_pr_curves")
    print("  Skipped: insufficient data")


SECTION 7: ROC and Precision-Recall Curves
  Saved: ROC and PR curves


In [9]:
"""
=============================================================================
SECTION 8: Ablation and Cross-Magnification Figures
=============================================================================
"""

print("=" * 70)
print("SECTION 8: Ablation and Cross-Magnification")
print("=" * 70)

# --- Ablation comparison ---
abl_path = os.path.join(RESULTS_DIR, "stage4_ablation_table.csv")
if os.path.exists(abl_path):
    abl = pd.read_csv(abl_path)
    test_abl = abl[abl['split'] == 'test']
    if len(test_abl) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        strategies = test_abl['strategy'].values
        x = np.arange(len(strategies))
        w = 0.35
        ax.bar(x - w/2, test_abl['macro_f1'].values, w,
               label='Macro-F1', color='#2E86AB')
        if 'balanced_accuracy' in test_abl.columns:
            ax.bar(x + w/2, test_abl['balanced_accuracy'].values, w,
                   label='Balanced Accuracy', color='#A23B72')
        ax.set_xticks(x)
        ax.set_xticklabels(strategies, rotation=45, ha='right')
        ax.set_ylabel('Score')
        ax.set_title('Aggregation Strategy Comparison\n'
                     '(Test, n=11, exploratory)')
        ax.legend()
        ax.set_ylim(0, 1.08)
        fig.tight_layout()
        for ext in ['png', 'pdf']:
            p = os.path.join(OUTPUT_DIR, f"fig_ablation.{ext}")
            fig.savefig(p)
        plt.close(fig)
        GENERATED.append("fig_ablation.png/pdf")
        print("  Saved: ablation comparison")
else:
    SKIPPED.append("fig_ablation")
    print("  Skipped: ablation table not found")


# --- Cross-magnification probability profiles ---
img_path = os.path.join(RESULTS_DIR, "stage3c_all_image_predictions.csv")
if os.path.exists(img_path):
    img_df = pd.read_csv(img_path)
    prob_col = 'swin_cal_prob'
    if prob_col in img_df.columns:
        test_img = img_df[img_df['split'] == 'test']
        if len(test_img) > 0:
            fig, ax = plt.subplots(figsize=(8, 5))
            for _, grp in test_img.groupby('patient_id'):
                vals, mags = [], []
                for m in MAGNIFICATIONS:
                    sub = grp[grp['magnification'] == m]
                    if len(sub) > 0:
                        vals.append(sub[prob_col].mean())
                        mags.append(m)
                color = '#B5542A' if grp['binary_label'].iloc[0] == 1 else '#1B7A72'
                ax.plot(mags, vals, 'o-', color=color, alpha=0.6, ms=5)
            ax.axhline(0.5, color='gray', ls='--', alpha=0.5)
            ax.set_xlabel('Magnification')
            ax.set_ylabel('Mean Calibrated P(malignant)')
            ax.set_title('Cross-Magnification Probability Profiles\n'
                         '(Test, n=11, exploratory)')
            ax.legend(handles=[
                mpatches.Patch(color='#1B7A72', label='Benign'),
                mpatches.Patch(color='#B5542A', label='Malignant'),
            ])
            fig.tight_layout()
            for ext in ['png', 'pdf']:
                p = os.path.join(OUTPUT_DIR, f"fig_cross_mag_profiles.{ext}")
                fig.savefig(p)
            plt.close(fig)
            GENERATED.append("fig_cross_mag_profiles.png/pdf")
            print("  Saved: cross-magnification profiles")
    else:
        print(f"  Skipped cross-mag: column '{prob_col}' not in image predictions")
        SKIPPED.append("fig_cross_mag_profiles")
else:
    SKIPPED.append("fig_cross_mag_profiles")
    print("  Skipped: stage3c image predictions not found")


SECTION 8: Ablation and Cross-Magnification
  Saved: ablation comparison
  Saved: cross-magnification profiles


In [10]:
"""
=============================================================================
SECTION 9: Output Summary
=============================================================================
"""

elapsed = time.time() - START_TIME

print("\n" + "=" * 70)
print("FINAL OUTPUT SUMMARY")
print("=" * 70)

print(f"\n  Total runtime: {elapsed:.1f} seconds")

print(f"\n  Files generated ({len(GENERATED)}):")
for f in GENERATED:
    print(f"    {f}")

if SKIPPED:
    print(f"\n  Skipped (inputs unavailable) ({len(SKIPPED)}):")
    for f in SKIPPED:
        print(f"    {f}")
else:
    print("\n  Nothing skipped. All inputs were available.")

print(f"\n  Output directory: {OUTPUT_DIR}")
print(f"\n  Contents:")
if os.path.isdir(OUTPUT_DIR):
    for f in sorted(os.listdir(OUTPUT_DIR)):
        sz = os.path.getsize(os.path.join(OUTPUT_DIR, f))
        print(f"    {f:50s} ({sz:>8,} bytes)")

print("\n" + "-" * 70)
print("IMPORTANT CAVEATS:")
print("  1. All results are EXPLORATORY (fixed split, 11 test patients).")
print("  2. Patient-aware cross-validation remains INCOMPLETE.")
print("  3. Bootstrap CIs on 11 patients are necessarily wide.")
print("  4. No clinical validation is claimed.")
print("  5. The learned aggregator's perfect test classification")
print("     on 11 patients should not be over-interpreted.")
print("-" * 70)



FINAL OUTPUT SUMMARY

  Total runtime: 203.3 seconds

  Files generated (10):
    table2_baseline_comparison.csv
    table4_learned_aggregator.csv
    table_bootstrap_cis.csv
    fig_confusion_matrices.png/pdf
    fig_reliability_resnet50.png/pdf
    fig_reliability_swin_small.png/pdf
    fig_reliability_learned.png/pdf
    fig_roc_pr_curves.png/pdf
    fig_ablation.png/pdf
    fig_cross_mag_profiles.png/pdf

  Nothing skipped. All inputs were available.

  Output directory: /content/drive/MyDrive/BreakHis_Project/final_paper_outputs

  Contents:
    fig_ablation.pdf                                   (  16,919 bytes)
    fig_ablation.png                                   ( 164,972 bytes)
    fig_confusion_matrices.pdf                         (  23,027 bytes)
    fig_confusion_matrices.png                         ( 125,974 bytes)
    fig_cross_mag_profiles.pdf                         (  19,527 bytes)
    fig_cross_mag_profiles.png                         ( 238,355 bytes)
    fig_reliab